In [1]:
# ============================================================
# THREE-PART NIGHTTIME THERMAL DATA HARMONISATION PIPELINE
# - Extracts annual environmental predictors by INSEE 200 m cell
# - Adds distance-to-water and INSEE socioeconomic indicators
# - Merges these covariates with the revised ECOSTRESS thermal outcomes
# - Exports harmonized GeoPackage, CSV, and Parquet datasets
#
# >>> THREE-PART REVISION:
#     * Preserves late_night and early_morning without transformation.
#     * Preserves cooling_magnitude_raw and NCD_raw.
#     * Keeps NCD as an exact alias of NCD_raw for compatibility.
#     * Preserves NCD_winsorized only as an optional sensitivity outcome.
#     * Performs identity checks before and after the merge.
# ============================================================


In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterstats import zonal_stats
from pathlib import Path
import os


In [3]:
raster_folder = "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/data/summer_ndvi_ndbi_dem_2154"

# Load grid
grid = gpd.read_file("C:/Users/kbonsu/Desktop/AICC Conference/idf_200_grid.shp")


In [4]:
grid.columns

Index(['fid', 'idcar_200m', 'idcar_1km', 'idcar_nat', 'i_est_200', 'i_est_1km',
       'lcog_geo', 'ind', 'men', 'men_pauv', 'men_1ind', 'men_5ind',
       'men_prop', 'men_fmp', 'ind_snv', 'men_surf', 'men_coll', 'men_mais',
       'log_av45', 'log_45_70', 'log_70_90', 'log_ap90', 'log_inc', 'log_soc',
       'ind_0_3', 'ind_4_5', 'ind_6_10', 'ind_11_17', 'ind_18_24', 'ind_25_39',
       'ind_40_54', 'ind_55_64', 'ind_65_79', 'ind_80p', 'ind_inc',
       'geometry'],
      dtype='str')

In [5]:
grid_proj = grid.copy()  # already in same CRS
grid_proj = grid_proj[grid_proj.is_valid]
grid_proj['geometry'] = grid_proj.buffer(0)

grid_ids = grid_proj['idcar_200m'].values


In [6]:
dem_raster = os.path.join(
    raster_folder,
    "UrbanPredictors_Summer_2018_2154.tif"
)

print("Extracting DEM once...")

dem_stats = zonal_stats(
    grid_proj,
    dem_raster,
    stats=['mean'],
    band=3,
    geojson_out=False
)

dem_values = [s['mean'] for s in dem_stats]


Extracting DEM once...


c:\Users\kbonsu\AppData\Local\miniconda3\envs\UHI\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


In [7]:
all_years = []

for year in range(2018, 2026):
    if year == 2024:
        # >>> MODIFIED: 2024 is skipped because no valid July ECOSTRESS NCD dataset is available
        continue

    raster_path = os.path.join(
        raster_folder,
        f"UrbanPredictors_Summer_{year}_2154.tif"
    )

    print(f"Processing {year}...")

    with rasterio.open(raster_path) as src:

        # NDVI (band 1)
        ndvi_stats = zonal_stats(
            grid_proj,
            src.read(1),
            affine=src.transform,
            stats=['mean'],
            geojson_out=False
        )

        # NDBI (band 2)
        ndbi_stats = zonal_stats(
            grid_proj,
            src.read(2),
            affine=src.transform,
            stats=['mean'],
            geojson_out=False
        )

    temp = pd.DataFrame({
        'grid_id': grid_ids,
        'year': year,
        'ndvi': [s['mean'] for s in ndvi_stats],
        'ndbi': [s['mean'] for s in ndbi_stats],
        'dem': dem_values   # reuse precomputed DEM
    })

    all_years.append(temp)

Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2025...


In [8]:
ndvi_panel = pd.concat(all_years, ignore_index=True)

print("Done.")
print(ndvi_panel.head())

Done.
                          grid_id  year      ndvi      ndbi         dem
0  CRS3035RES200mN2799400E3840400  2018       NaN       NaN         NaN
1  CRS3035RES200mN2799400E3840600  2018       NaN       NaN         NaN
2  CRS3035RES200mN2799800E3840000  2018  0.311374  0.035417  229.286499
3  CRS3035RES200mN2800000E3838600  2018  0.520237 -0.190765  230.055710
4  CRS3035RES200mN2800000E3840000  2018  0.338598  0.036139  225.800618


In [9]:
ndvi_panel.shape

(652001, 5)

In [10]:
dist = pd.read_csv("C:/Users/kbonsu/Desktop/AICC Conference/idf_distance_to_water.csv")

dist.head()

,idcar_200m,DistWater,DistWater_log
0,CRS3035RES200mN2799400E3840400,38422.015810,10.556412
1,CRS3035RES200mN2799400E3840600,38484.062882,10.558025
2,CRS3035RES200mN2799800E3840000,37973.642326,10.544674
3,CRS3035RES200mN2800000E3838600,37314.924011,10.527175
4,CRS3035RES200mN2800000E3840000,37786.807275,10.539742


In [11]:
dist.shape

(93143, 3)

In [12]:
dist = dist.rename(columns={'idcar_200m': 'grid_id'})
dist['grid_id'] = dist['grid_id'].astype(str)
ndvi_panel['grid_id'] = ndvi_panel['grid_id'].astype(str)

In [13]:
ndvi_panel = ndvi_panel.merge(
    dist[['grid_id', 'DistWater', 'DistWater_log']],
    on='grid_id',
    how='left'
)


In [14]:
ndvi_panel.shape

(652001, 7)

In [15]:
ndvi_panel[['DistWater','DistWater_log']].isna().mean()


DistWater        0.0
DistWater_log    0.0
dtype: float64

In [16]:
grid = grid.rename(columns={'idcar_200m': 'grid_id'})
grid['grid_id'] = grid['grid_id'].astype(str)

In [17]:
# =============================================================================
# SAFE DENOMINATORS — avoid division by zero
# =============================================================================
# We replace 0 with NaN so derived shares become NaN rather than inf/0.
# This propagates missingness correctly through all downstream variables.

grid["ind"] = grid["ind"].replace(0, np.nan)
grid["men"] = grid["men"].replace(0, np.nan)

# Total dated dwellings (used as denominator for age shares + social share)
grid["total_dwellings"]  = (
    grid["log_av45"] +
    grid["log_45_70"] +
    grid["log_70_90"] +
    grid["log_ap90"]
)
grid["total_dwellings"]  = grid["total_dwellings"].replace(0, np.nan)

# =============================================================================
# 1. ECONOMIC STRUCTURE
# =============================================================================

# Mean living standard (Winsorized sum of individual living standards / N)
# ind_snv is already Winsorized by INSEE — no further trimming needed
grid["mean_income"] = grid["ind_snv"] / grid["ind"]

# Poverty rate: share of poor households among all households
grid["poverty_rate"] = grid["men_pauv"] / grid["men"]

# =============================================================================
# 2. POPULATION DENSITY
# =============================================================================
# 200m cell area = 200m × 200m = 40,000 m² = 0.04 km²

grid["pop_density"] = grid["ind"] / 0.04   # individuals per km²

# =============================================================================
# 3. HOUSING STRUCTURE
# =============================================================================

# Share of households in collective (apartment) housing
grid["share_collective"] = grid["men_coll"] / grid["men"]

# Share of households in individual houses
# NOTE: share_house ≈ 1 - share_collective in most cells;
# include only if you want to explicitly model the house/apartment split.
# Drop from regression if VIF is high.
grid["share_house"]      = grid["men_mais"] / grid["men"]

# Mean dwelling surface area per household (m²)
# Proxy for housing quality and crowding — smaller = more crowded = higher risk
grid["mean_dwelling_surface"] = grid["men_surf"] / grid["men"]

# Share of large households (5+ individuals) — crowding indicator
grid["large_hh_share"]   = grid["men_5ind"] / grid["men"]

# =============================================================================
# 4. BUILDING AGE SHARES
# =============================================================================
# Reference category: share_post90 (newest stock, best insulated)
# → omit from regression to avoid perfect multicollinearity
# >>> MODIFIED INTERPRETATION:
# For the NCD outcome, positive coefficients indicate an association with
# weaker nighttime cooling; negative coefficients indicate stronger cooling.
# Building-age mechanisms remain hypotheses rather than directly measured processes.

grid["share_pre45"]      = grid["log_av45"]  / grid["total_dwellings"]
grid["share_45_70"]      = grid["log_45_70"] / grid["total_dwellings"]
grid["share_70_90"]      = grid["log_70_90"] / grid["total_dwellings"]
grid["share_post90"]     = grid["log_ap90"]  / grid["total_dwellings"]
# share_post90 is the reference — exclude from model, keep for QC only

# Quick sanity check: shares should sum to ~1 (within rounding + log_inc)
share_sum = (
    grid["share_pre45"].fillna(0) +
    grid["share_45_70"].fillna(0) +
    grid["share_70_90"].fillna(0) +
    grid["share_post90"].fillna(0)
)
print(f"\nBuilding age share sum — mean: {share_sum.mean():.3f}  "
      f"(should be ≈1.0 where log_inc=0)")

# =============================================================================
# 5. SOCIAL HOUSING
# =============================================================================
# Social housing share among all dated dwellings
grid["social_share"]     = grid["log_soc"] / grid["total_dwellings"]

# =============================================================================
# 6. DEMOGRAPHIC VULNERABILITY
# =============================================================================

# Elderly share — 65+ (French public health standard; used in canicule research)
# Your previous definition used 80+ only; 65+ is the standard heat-risk threshold
grid["elderly_share"]    = (
    grid["ind_65_79"] + grid["ind_80p"]
) / grid["ind"]

# Very elderly share — 80+ separately, for sensitivity analysis
grid["very_elderly_share"] = grid["ind_80p"] / grid["ind"]

# Child share — 0 to 5 years (heat-vulnerable age group)
grid["child_share"]      = (
    grid["ind_0_3"] + grid["ind_4_5"]
) / grid["ind"]

# Broader child share — 0 to 17 years (alternative specification)
grid["child_broad_share"] = (
    grid["ind_0_3"]  +
    grid["ind_4_5"]  +
    grid["ind_6_10"] +
    grid["ind_11_17"]
) / grid["ind"]

# Single-parent household share — social vulnerability indicator
grid["single_parent_share"] = grid["men_fmp"] / grid["men"]

# Single-person household share — social isolation / no cooling support
grid["single_person_share"] = grid["men_1ind"] / grid["men"]




Building age share sum — mean: 1.000  (should be ≈1.0 where log_inc=0)


In [18]:
grid_socio = grid[[
    "grid_id",
    "total_dwellings",
    "mean_income",
    "poverty_rate",
    "pop_density",
    "share_collective",
    "share_house",
    "mean_dwelling_surface",
    "large_hh_share",
    "share_pre45",
    "share_45_70",
    "share_70_90",
    "share_post90",
    "social_share",
    "elderly_share",
    "very_elderly_share",
    "child_share",
    "child_broad_share",
    "single_parent_share",
    "single_person_share"
]]



In [19]:
ndvi_panel['grid_id'] = ndvi_panel['grid_id'].astype(str)

ndvi_panel = ndvi_panel.merge(
    grid_socio,
    on='grid_id',
    how='left'
)


In [20]:
ndvi_panel.shape

(652001, 26)

In [21]:
ndvi_panel.describe()

,year,ndvi,ndbi,dem,DistWater,DistWater_log,total_dwellings,mean_income,poverty_rate,pop_density,...,share_45_70,share_70_90,share_post90,social_share,elderly_share,very_elderly_share,child_share,child_broad_share,single_parent_share,single_person_share
count,652001.000000,651623.000000,651623.000000,651623.000000,652001.000000,652001.000000,651987.000000,652001.000000,652001.000000,652001.000000,...,651987.000000,651987.000000,651987.000000,651987.000000,652001.000000,652001.000000,652001.000000,652001.000000,652001.000000,652001.000000
mean,2021.142857,0.519707,-0.112574,96.633250,6934.924550,8.059640,56.636769,27996.175463,0.085196,3402.701894,...,0.158083,0.281828,0.272720,0.066878,0.176730,0.045498,0.065768,0.223125,0.094046,0.248275
std,2.231502,0.163320,0.117526,41.243654,8001.165623,1.473718,122.809224,6418.463993,0.085285,6815.281660,...,0.197162,0.253138,0.245709,0.198510,0.089697,0.040581,0.040095,0.071385,0.068678,0.129375
min,2018.000000,-0.800625,-0.670565,9.221630,0.112900,0.106970,0.200000,8646.413953,0.000000,25.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2019.000000,0.404696,-0.191157,63.602696,1228.877645,7.114670,3.100000,23880.659259,0.021739,200.000000,...,0.000000,0.086207,0.090909,0.000000,0.114286,0.017647,0.038835,0.178571,0.047619,0.157143
50%,2021.000000,0.511695,-0.093691,92.531169,3912.868659,8.272282,13.000000,27098.457292,0.066667,875.000000,...,0.095238,0.226415,0.210526,0.000000,0.166667,0.038095,0.063380,0.221477,0.086957,0.237500
75%,2023.000000,0.636395,-0.023072,128.212570,9346.636474,9.142879,52.000000,31146.213095,0.122727,3400.000000,...,0.220930,0.400000,0.375000,0.000000,0.225000,0.064706,0.091358,0.268000,0.133333,0.325806
max,2025.000000,0.960788,0.865816,243.542550,51084.752916,10.841261,2204.000000,79961.300000,0.800000,164137.500000,...,1.000000,1.000000,1.000000,1.517241,1.000000,0.893617,0.305882,0.550000,0.576923,1.000000


In [22]:
# >>> THREE-PART REVISION: read the revised preprocessing output
THERMAL_INPUT = (
    "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/"
    "ecostress_paris_july_three_part_thermal_outcomes_200m.gpkg"
)

thermal_july = gpd.read_file(THERMAL_INPUT)

required_thermal_columns = {
    "idcar_200m", "year", "month",
    "late_night", "early_morning",
    "n_late_night", "n_early_morning",
    "cooling_magnitude_raw", "NCD_raw", "NCD",
    "NCD_winsorized", "geometry"
}
missing_thermal_columns = required_thermal_columns.difference(thermal_july.columns)
if missing_thermal_columns:
    raise ValueError(
        "Revised preprocessing output is missing required columns: "
        f"{sorted(missing_thermal_columns)}"
    )

# Validate before any merge
assert np.allclose(
    thermal_july["NCD_raw"],
    thermal_july["early_morning"] - thermal_july["late_night"],
    equal_nan=True
)
assert np.allclose(
    thermal_july["cooling_magnitude_raw"],
    thermal_july["late_night"] - thermal_july["early_morning"],
    equal_nan=True
)
assert np.allclose(
    thermal_july["NCD_raw"],
    -thermal_july["cooling_magnitude_raw"],
    equal_nan=True
)
assert np.allclose(
    thermal_july["NCD"],
    thermal_july["NCD_raw"],
    equal_nan=True
)

print("Loaded and validated revised three-part thermal dataset:", thermal_july.shape)
thermal_july.head()


Loaded and validated revised three-part thermal dataset: (57813, 12)


,idcar_200m,year,month,late_night,early_morning,n_late_night,n_early_morning,cooling_magnitude_raw,NCD_raw,NCD,NCD_winsorized,geometry
0,CRS3035RES200mN2869400E3773400,2019,7,17.317142,15.593746,3.0,6.0,1.723396,-1.723396,-1.723396,-1.723396,"MULTIPOLYGON (((667098.621 6843477.207, 667109..."
1,CRS3035RES200mN2869400E3773400,2022,7,18.901672,13.553332,3.0,3.0,5.348340,-5.348340,-5.348340,-5.348340,"MULTIPOLYGON (((667098.621 6843477.207, 667109..."
2,CRS3035RES200mN2869400E3773400,2023,7,19.197365,18.387997,4.0,2.0,0.809368,-0.809368,-0.809368,-0.809368,"MULTIPOLYGON (((667098.621 6843477.207, 667109..."
3,CRS3035RES200mN2869400E3773600,2019,7,15.899229,15.123752,3.0,6.0,0.775477,-0.775477,-0.775477,-0.775477,"MULTIPOLYGON (((667297.791 6843496.26, 667306...."
4,CRS3035RES200mN2869400E3773600,2022,7,18.921541,14.201250,4.0,3.0,4.720291,-4.720291,-4.720291,-4.720291,"MULTIPOLYGON (((667297.791 6843496.26, 667306...."


In [23]:
thermal_july.shape


(57813, 12)

In [24]:
# >>> THREE-PART REVISION: harmonize identifiers and merge with annual covariates
thermal_july = thermal_july.rename(columns={"idcar_200m": "grid_id"})

thermal_july["grid_id"] = thermal_july["grid_id"].astype(str)
ndvi_panel["grid_id"] = ndvi_panel["grid_id"].astype(str)
thermal_july["year"] = thermal_july["year"].astype(int)
ndvi_panel["year"] = ndvi_panel["year"].astype(int)

harmonized_panel = thermal_july.merge(
    ndvi_panel,
    on=["grid_id", "year"],
    how="left",
    validate="many_to_one"
)

print("Harmonized three-part panel created.")
print("Raw thermal outcomes were preserved without recalculation or clipping.")


Harmonized three-part panel created.
Raw thermal outcomes were preserved without recalculation or clipping.


In [25]:
# >>> MODIFIED: inspect the harmonized NCD analytical panel
harmonized_panel.shape


(57813, 36)

In [26]:
# >>> THREE-PART REVISION: final quality-control checks
print("Rows:", len(harmonized_panel))
print("Unique grid cells:", harmonized_panel["grid_id"].nunique())
print("Observations by year:")
print(harmonized_panel.groupby("year").size())

key_cols = [
    "late_night", "early_morning",
    "cooling_magnitude_raw", "NCD_raw", "NCD", "NCD_winsorized",
    "ndvi", "dem"
]
print("\nMissing-value shares for core variables:")
print(harmonized_panel[key_cols].isna().mean().sort_values(ascending=False))

# Mathematical identities must survive the merge exactly
assert np.allclose(
    harmonized_panel["NCD_raw"],
    harmonized_panel["early_morning"] - harmonized_panel["late_night"],
    equal_nan=True
)
assert np.allclose(
    harmonized_panel["cooling_magnitude_raw"],
    harmonized_panel["late_night"] - harmonized_panel["early_morning"],
    equal_nan=True
)
assert np.allclose(
    harmonized_panel["NCD_raw"],
    -harmonized_panel["cooling_magnitude_raw"],
    equal_nan=True
)
assert np.allclose(
    harmonized_panel["NCD"],
    harmonized_panel["NCD_raw"],
    equal_nan=True
)

print("\nAll thermal identities verified after harmonisation.")
print("\nRaw NCD summary:")
print(harmonized_panel["NCD_raw"].describe())


Rows: 57813
Unique grid cells: 13964
Observations by year:
year
2019    13963
2021     2766
2022    13953
2023    13732
2025    13399
dtype: int64

Missing-value shares for core variables:
late_night               0.0
early_morning            0.0
cooling_magnitude_raw    0.0
NCD_raw                  0.0
NCD                      0.0
NCD_winsorized           0.0
ndvi                     0.0
dem                      0.0
dtype: float64

All thermal identities verified after harmonisation.

Raw NCD summary:
count    57813.000000
mean        -3.678240
std          1.952358
min        -11.100002
25%         -5.160683
50%         -3.511856
75%         -2.204439
max          2.836340
Name: NCD_raw, dtype: float64


In [27]:
# >>> MODIFIED: export harmonized NCD GeoPackage
harmonized_panel.to_file(
    "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/"
    "july_paris_three_part_thermal_harmonized.gpkg",
    driver="GPKG",
    index=False
)


In [28]:
# >>> MODIFIED: export harmonized NCD CSV
harmonized_panel.drop(columns='geometry').to_csv(
    "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/"
    "july_paris_three_part_thermal_harmonized.csv",
    index=False
)


In [29]:
# >>> MODIFIED: export harmonized NCD Parquet with geometry retained
harmonized_panel.to_parquet(
    "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/"
    "july_paris_three_part_thermal_harmonized.parquet",
    index=False
)


In [30]:
# >>> MODIFIED: years retained in the harmonized NCD panel
sorted(harmonized_panel.year.dropna().unique())


[np.int64(2019),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2025)]

In [31]:
# >>> MODIFIED: verify that NCD and its component temperature-window variables are retained
harmonized_panel.columns.tolist()


['grid_id',
 'year',
 'month',
 'late_night',
 'early_morning',
 'n_late_night',
 'n_early_morning',
 'cooling_magnitude_raw',
 'NCD_raw',
 'NCD',
 'NCD_winsorized',
 'geometry',
 'ndvi',
 'ndbi',
 'dem',
 'DistWater',
 'DistWater_log',
 'total_dwellings',
 'mean_income',
 'poverty_rate',
 'pop_density',
 'share_collective',
 'share_house',
 'mean_dwelling_surface',
 'large_hh_share',
 'share_pre45',
 'share_45_70',
 'share_70_90',
 'share_post90',
 'social_share',
 'elderly_share',
 'very_elderly_share',
 'child_share',
 'child_broad_share',
 'single_parent_share',
 'single_person_share']

In [32]:
# >>> MODIFIED: preview final harmonized NCD dataset
harmonized_panel.head()


,grid_id,year,month,late_night,early_morning,n_late_night,n_early_morning,cooling_magnitude_raw,NCD_raw,NCD,...,share_45_70,share_70_90,share_post90,social_share,elderly_share,very_elderly_share,child_share,child_broad_share,single_parent_share,single_person_share
0,CRS3035RES200mN2869400E3773400,2019,7,17.317142,15.593746,3.0,6.0,1.723396,-1.723396,-1.723396,...,0.0,0.4,0.6,0.0,0.05000,0.025,0.025000,0.300000,0.090909,0.090909
1,CRS3035RES200mN2869400E3773400,2022,7,18.901672,13.553332,3.0,3.0,5.348340,-5.348340,-5.348340,...,0.0,0.4,0.6,0.0,0.05000,0.025,0.025000,0.300000,0.090909,0.090909
2,CRS3035RES200mN2869400E3773400,2023,7,19.197365,18.387997,4.0,2.0,0.809368,-0.809368,-0.809368,...,0.0,0.4,0.6,0.0,0.05000,0.025,0.025000,0.300000,0.090909,0.090909
3,CRS3035RES200mN2869400E3773600,2019,7,15.899229,15.123752,3.0,6.0,0.775477,-0.775477,-0.775477,...,0.0,0.0,1.0,0.0,0.07438,0.000,0.033058,0.173554,0.050000,0.150000
4,CRS3035RES200mN2869400E3773600,2022,7,18.921541,14.201250,4.0,3.0,4.720291,-4.720291,-4.720291,...,0.0,0.0,1.0,0.0,0.07438,0.000,0.033058,0.173554,0.050000,0.150000


In [33]:
import rasterio

with rasterio.open("C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/data/summer_ndvi_ndbi_dem_2154/UrbanPredictors_Summer_2019_2154.tif") as src:
    print("Number of bands:", src.count)
    print("Band descriptions:", src.descriptions)

Number of bands: 5
Band descriptions: (None, None, None, None, None)
